# Data Inventory EDA

This notebook reads every clean CSV in `data/processed`, checks schema, null coverage, pandas dtypes, duplicate URLs, and basic salary/skill coverage.

TopDev is kept in the full inventory and demand exploration. It is excluded only from numeric salary analysis when no `salary_min` or `salary_max` exists.

In [1]:
from pathlib import Path
import ast
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 180)

try:
    from IPython.display import display
except Exception:  # pragma: no cover - notebook convenience fallback
    def display(obj):
        if isinstance(obj, pd.DataFrame):
            print(obj.to_string(index=False))
        else:
            print(obj)


def find_repo_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "processed").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing data/processed")


REPO_ROOT = find_repo_root()
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
REPORT_DIR = REPO_ROOT / "data" / "reports"

print(f"Repo root: {REPO_ROOT}")
print(f"Processed data dir: {PROCESSED_DIR}")

Repo root: G:\project\Vietnam IT Job Market Intelligence
Processed data dir: G:\project\Vietnam IT Job Market Intelligence\data\processed


## Load all clean CSVs

The load step keeps `_input_file` so sample/test/full/salary runs can be inspected separately before deduping.

In [2]:
clean_paths = sorted(PROCESSED_DIR.glob("*_clean.csv"))
if not clean_paths:
    raise FileNotFoundError(f"No *_clean.csv files found in {PROCESSED_DIR}")

frames_by_file = {}
for path in clean_paths:
    frame = pd.read_csv(path, encoding="utf-8-sig")
    frame["_input_file"] = path.name
    frames_by_file[path.name] = frame

jobs = pd.concat(frames_by_file.values(), ignore_index=True, sort=False)
base_columns = [column for column in next(iter(frames_by_file.values())).columns if column != "_input_file"]

numeric_columns = ["salary_min", "salary_max", "experience_min", "experience_max"]
for column in numeric_columns:
    if column in jobs.columns:
        jobs[column] = pd.to_numeric(jobs[column], errors="coerce")

date_columns = ["scraped_at", "posted_at", "valid_through"]
for column in date_columns:
    if column in jobs.columns:
        jobs[f"{column}_parsed"] = pd.to_datetime(jobs[column], errors="coerce", utc=True)

jobs["_has_numeric_salary"] = jobs[["salary_min", "salary_max"]].notna().any(axis=1)
jobs["_salary_midpoint"] = jobs[["salary_min", "salary_max"]].mean(axis=1)
jobs["_has_skills"] = jobs["skills"].fillna("").astype(str).str.strip().ne("")
jobs["_has_experience"] = jobs["experience_raw"].fillna("").astype(str).str.strip().ne("")
jobs["_has_description"] = jobs["description"].fillna("").astype(str).str.strip().ne("")
jobs["_quality_score"] = (
    jobs["_has_numeric_salary"].astype(int) * 100
    + jobs["_has_skills"].astype(int) * 10
    + jobs["_has_experience"].astype(int) * 5
    + jobs["_has_description"].astype(int)
)

print(f"Clean CSV files loaded: {len(clean_paths)}")
print(f"Rows loaded: {len(jobs):,}")
print(f"Unique URLs: {jobs['url'].nunique():,}")
display(pd.DataFrame({"input_file": [path.name for path in clean_paths]}))

Clean CSV files loaded: 8
Rows loaded: 1,662
Unique URLs: 1,433


,input_file
0,itviec_20260701_100_clean.csv
1,itviec_salary_20260702_clean.csv
2,itviec_sample_50_clean.csv
3,itviec_test_2_clean.csv
4,topcv_20260702_100_clean.csv
5,topcv_20260702_test_5_clean.csv
6,topcv_salary_20260702_clean.csv
7,topdev_20260701_100_clean.csv


## Inventory by file and source

In [3]:
def non_empty_count(series):
    return int(series.fillna("").astype(str).str.strip().ne("").sum())


inventory = (
    jobs.groupby(["_input_file", "source"], dropna=False)
    .agg(
        rows=("url", "size"),
        unique_urls=("url", "nunique"),
        parse_ok_rows=("parse_status", lambda s: int(s.astype(str).eq("ok").sum())),
        numeric_salary_rows=("_has_numeric_salary", "sum"),
        title_non_empty=("title", non_empty_count),
        company_non_empty=("company", non_empty_count),
        location_non_empty=("location", non_empty_count),
        salary_raw_non_empty=("salary_raw", non_empty_count),
        skills_non_empty=("skills", non_empty_count),
        experience_non_empty=("experience_raw", non_empty_count),
        description_non_empty=("description", non_empty_count),
    )
    .reset_index()
)
inventory["parse_ok_rate"] = inventory["parse_ok_rows"] / inventory["rows"]
inventory["salary_numeric_fill_rate"] = inventory["numeric_salary_rows"] / inventory["rows"]
inventory["skills_fill_rate"] = inventory["skills_non_empty"] / inventory["rows"]
inventory["experience_fill_rate"] = inventory["experience_non_empty"] / inventory["rows"]

display(inventory.sort_values(["source", "_input_file"]).reset_index(drop=True))

source_inventory = (
    jobs.groupby("source", dropna=False)
    .agg(
        rows=("url", "size"),
        unique_urls=("url", "nunique"),
        numeric_salary_rows=("_has_numeric_salary", "sum"),
        parse_ok_rows=("parse_status", lambda s: int(s.astype(str).eq("ok").sum())),
        salary_raw_non_empty=("salary_raw", non_empty_count),
        skills_non_empty=("skills", non_empty_count),
        experience_non_empty=("experience_raw", non_empty_count),
    )
    .reset_index()
)
source_inventory["salary_numeric_fill_rate"] = source_inventory["numeric_salary_rows"] / source_inventory["rows"]
source_inventory["parse_ok_rate"] = source_inventory["parse_ok_rows"] / source_inventory["rows"]
display(source_inventory.sort_values("source"))

,_input_file,source,rows,unique_urls,parse_ok_rows,numeric_salary_rows,title_non_empty,company_non_empty,location_non_empty,salary_raw_non_empty,skills_non_empty,experience_non_empty,description_non_empty,parse_ok_rate,salary_numeric_fill_rate,skills_fill_rate,experience_fill_rate
0,itviec_20260701_100_clean.csv,itviec,100,100,100,27,100,100,100,100,100,82,100,1.0,0.270000,1.000000,0.820000
1,itviec_salary_20260702_clean.csv,itviec,750,750,750,207,750,750,750,750,750,658,750,1.0,0.276000,1.000000,0.877333
2,itviec_sample_50_clean.csv,itviec,30,30,30,8,30,30,30,30,30,25,30,1.0,0.266667,1.000000,0.833333
3,itviec_test_2_clean.csv,itviec,2,2,2,0,2,2,2,2,2,1,2,1.0,0.000000,1.000000,0.500000
4,topcv_20260702_100_clean.csv,topcv,100,100,100,55,100,100,100,95,77,81,100,1.0,0.550000,0.770000,0.810000
5,topcv_20260702_test_5_clean.csv,topcv,5,5,5,2,5,5,5,5,4,4,5,1.0,0.400000,0.800000,0.800000
6,topcv_salary_20260702_clean.csv,topcv,575,575,575,298,575,575,575,526,446,488,575,1.0,0.518261,0.775652,0.848696
7,topdev_20260701_100_clean.csv,topdev,100,100,100,0,100,100,100,100,100,83,100,1.0,0.000000,1.000000,0.830000


,source,rows,unique_urls,numeric_salary_rows,parse_ok_rows,salary_raw_non_empty,skills_non_empty,experience_non_empty,salary_numeric_fill_rate,parse_ok_rate
0,itviec,882,750,242,882,882,882,766,0.274376,1.0
1,topcv,680,583,355,680,626,527,573,0.522059,1.0
2,topdev,100,100,0,100,100,100,83,0.000000,1.0


## Schema, dtypes, and parsed dates

Raw CSV date columns read as `object`; the notebook adds parsed UTC datetime columns for EDA without changing source files.

In [4]:
schema_rows = []
for file_name, frame in frames_by_file.items():
    for column in frame.columns:
        schema_rows.append({"_input_file": file_name, "column": column, "dtype": str(frame[column].dtype)})

schema_long = pd.DataFrame(schema_rows)
schema_presence = (
    schema_long.assign(present=True)
    .pivot_table(index="column", columns="_input_file", values="present", aggfunc="any", fill_value=False)
    .reset_index()
)
display(schema_presence)

dtype_matrix = schema_long.pivot(index="column", columns="_input_file", values="dtype").reset_index()
display(dtype_matrix)

dtype_checks = []
for column in numeric_columns:
    dtype_checks.append(
        {
            "column": column,
            "expected_for_eda": "numeric",
            "actual_dtype_after_load": str(jobs[column].dtype),
            "ok": pd.api.types.is_numeric_dtype(jobs[column]),
        }
    )
for column in date_columns:
    parsed = f"{column}_parsed"
    dtype_checks.append(
        {
            "column": parsed,
            "expected_for_eda": "datetime64[ns, UTC]",
            "actual_dtype_after_load": str(jobs[parsed].dtype),
            "ok": pd.api.types.is_datetime64_any_dtype(jobs[parsed]),
        }
    )
display(pd.DataFrame(dtype_checks))

date_parse_summary = []
for column in date_columns:
    parsed = f"{column}_parsed"
    date_parse_summary.append(
        {
            "column": column,
            "raw_non_empty": non_empty_count(jobs[column]),
            "parsed_non_null": int(jobs[parsed].notna().sum()),
            "parse_null_rate": round(float(jobs[parsed].isna().mean()), 4),
        }
    )
display(pd.DataFrame(date_parse_summary))

_input_file,column,itviec_20260701_100_clean.csv,itviec_salary_20260702_clean.csv,itviec_sample_50_clean.csv,itviec_test_2_clean.csv,topcv_20260702_100_clean.csv,topcv_20260702_test_5_clean.csv,topcv_salary_20260702_clean.csv,topdev_20260701_100_clean.csv
0,_input_file,True,True,True,True,True,True,True,True
1,company,True,True,True,True,True,True,True,True
2,description,True,True,True,True,True,True,True,True
3,employment_type,True,True,True,True,True,True,True,True
4,experience_max,True,True,True,True,True,True,True,True
5,experience_min,True,True,True,True,True,True,True,True
6,experience_raw,True,True,True,True,True,True,True,True
7,job_id,True,True,True,True,True,True,True,True
8,location,True,True,True,True,True,True,True,True
9,location_cities,True,True,True,True,True,True,True,True


_input_file,column,itviec_20260701_100_clean.csv,itviec_salary_20260702_clean.csv,itviec_sample_50_clean.csv,itviec_test_2_clean.csv,topcv_20260702_100_clean.csv,topcv_20260702_test_5_clean.csv,topcv_salary_20260702_clean.csv,topdev_20260701_100_clean.csv
0,_input_file,object,object,object,object,object,object,object,object
1,company,object,object,object,object,object,object,object,object
2,description,object,object,object,object,object,object,object,object
3,employment_type,object,object,object,object,object,object,object,object
4,experience_max,float64,float64,float64,float64,float64,float64,float64,float64
5,experience_min,float64,float64,float64,float64,float64,float64,float64,float64
6,experience_raw,object,object,object,object,object,object,object,object
7,job_id,object,object,object,object,object,object,object,object
8,location,object,object,object,object,object,object,object,object
9,location_cities,object,object,object,object,object,object,object,object


,column,expected_for_eda,actual_dtype_after_load,ok
0,salary_min,numeric,float64,True
1,salary_max,numeric,float64,True
2,experience_min,numeric,float64,True
3,experience_max,numeric,float64,True
4,scraped_at_parsed,"datetime64[ns, UTC]","datetime64[ns, UTC]",True
5,posted_at_parsed,"datetime64[ns, UTC]","datetime64[ns, UTC]",True
6,valid_through_parsed,"datetime64[ns, UTC]","datetime64[ns, UTC]",True


,column,raw_non_empty,parsed_non_null,parse_null_rate
0,scraped_at,1662,1662,0.0000
1,posted_at,1662,100,0.9398
2,valid_through,1662,100,0.9398


## Null and fill-rate checks

In [5]:
def null_summary(frame, columns):
    rows = []
    for column in columns:
        rows.append(
            {
                "column": column,
                "dtype": str(frame[column].dtype),
                "non_null": int(frame[column].notna().sum()),
                "non_empty": non_empty_count(frame[column]),
                "nulls": int(frame[column].isna().sum()),
                "null_rate": round(float(frame[column].isna().mean()), 4),
            }
        )
    return pd.DataFrame(rows)


overall_nulls = null_summary(jobs, [*base_columns, "_input_file"])
display(overall_nulls)

source_nulls = []
for source, group in jobs.groupby("source", dropna=False):
    summary = null_summary(group, base_columns)
    summary.insert(0, "source", source)
    source_nulls.append(summary)
source_nulls = pd.concat(source_nulls, ignore_index=True)
display(source_nulls.sort_values(["source", "null_rate", "column"], ascending=[True, False, True]).head(80))

,column,dtype,non_null,non_empty,nulls,null_rate
0,source,object,1662,1662,0,0.0000
1,url,object,1662,1662,0,0.0000
2,job_id,object,1662,1662,0,0.0000
3,title,object,1662,1662,0,0.0000
4,company,object,1662,1662,0,0.0000
5,location,object,1662,1662,0,0.0000
6,salary_raw,object,1608,1608,54,0.0325
7,salary_min,float64,569,569,1093,0.6576
8,salary_max,float64,584,584,1078,0.6486
9,salary_currency,object,592,592,1070,0.6438


,source,column,dtype,non_null,non_empty,nulls,null_rate
13,itviec,experience_max,float64,184,184,698,0.7914
7,itviec,salary_min,float64,216,216,666,0.7551
8,itviec,salary_max,float64,238,238,644,0.7302
9,itviec,salary_currency,object,240,240,642,0.7279
12,itviec,experience_min,float64,766,766,116,0.1315
...,...,...,...,...,...,...,...
60,topdev,skills,object,100,100,0,0.0000
50,topdev,source,object,100,100,0,0.0000
53,topdev,title,object,100,100,0,0.0000
51,topdev,url,object,100,100,0,0.0000


## Duplicate URL handling

EDA keeps both all-row and deduped-by-URL views. The deduped view keeps the row with the richest parsed fields for each URL.

In [6]:
duplicate_mask = jobs["url"].duplicated(keep=False)
duplicate_rows = jobs.loc[duplicate_mask].sort_values(["url", "_quality_score", "_input_file"])

print(f"Rows in duplicate URL groups: {len(duplicate_rows):,}")
print(f"Unique duplicated URLs: {duplicate_rows['url'].nunique():,}")
display(duplicate_rows[["source", "_input_file", "url", "title", "company", "_quality_score"]].head(50))

jobs_deduped = (
    jobs.sort_values(["url", "_quality_score", "_input_file"])
    .drop_duplicates("url", keep="last")
    .copy()
)

deduped_summary = (
    jobs_deduped.groupby("source", dropna=False)
    .agg(
        rows=("url", "size"),
        numeric_salary_rows=("_has_numeric_salary", "sum"),
        salary_raw_non_empty=("salary_raw", non_empty_count),
        skills_non_empty=("skills", non_empty_count),
        experience_non_empty=("experience_raw", non_empty_count),
    )
    .reset_index()
)
deduped_summary["salary_numeric_fill_rate"] = deduped_summary["numeric_salary_rows"] / deduped_summary["rows"]

print(f"Rows after URL dedupe: {len(jobs_deduped):,}")
display(deduped_summary.sort_values("source"))

Rows in duplicate URL groups: 421
Unique duplicated URLs: 192


,source,_input_file,url,title,company,_quality_score
12,itviec,itviec_20260701_100_clean.csv,https://itviec.com/it-jobs/agentic-engineer-python-go-c-c-backend-ai-agents-mb-bank-1315,Agentic Engineer (Python/Go/C/C++ Backend & AI Agents),MB Bank,16
112,itviec,itviec_salary_20260702_clean.csv,https://itviec.com/it-jobs/agentic-engineer-python-go-c-c-backend-ai-agents-mb-bank-1315,Agentic Engineer (Python/Go/C/C++ Backend & AI Agents),MB Bank,16
863,itviec,itviec_sample_50_clean.csv,https://itviec.com/it-jobs/agentic-engineer-python-go-c-c-backend-ai-agents-mb-bank-1315,Agentic Engineer (Python/Go/C/C++ Backend & AI Agents),MB Bank,16
49,itviec,itviec_20260701_100_clean.csv,https://itviec.com/it-jobs/ai-agent-engineer-ml-python-typesript-net-adk-truong-minh-thinh-technology-joint-stock-co...,"AI Agent Engineer (ML, Python, TypeSript, .NET, ADK)",TRUONG MINH THINH TECHNOLOGY JOINT STOCK COMPANY,16
149,itviec,itviec_salary_20260702_clean.csv,https://itviec.com/it-jobs/ai-agent-engineer-ml-python-typesript-net-adk-truong-minh-thinh-technology-joint-stock-co...,"AI Agent Engineer (ML, Python, TypeSript, .NET, ADK)",TRUONG MINH THINH TECHNOLOGY JOINT STOCK COMPANY,16
28,itviec,itviec_20260701_100_clean.csv,https://itviec.com/it-jobs/ai-agent-engineer-python-llm-kds-vietnam-5513,"AI Agent Engineer (Python, LLM)",KDS Vietnam,116
128,itviec,itviec_salary_20260702_clean.csv,https://itviec.com/it-jobs/ai-agent-engineer-python-llm-kds-vietnam-5513,"AI Agent Engineer (Python, LLM)",KDS Vietnam,116
16,itviec,itviec_20260701_100_clean.csv,https://itviec.com/it-jobs/ai-automation-engineer-javascript-python-llm-bloomentum-2901,"AI Automation Engineer (JavaScript, Python, LLM)",Bloomentum,16
116,itviec,itviec_salary_20260702_clean.csv,https://itviec.com/it-jobs/ai-automation-engineer-javascript-python-llm-bloomentum-2901,"AI Automation Engineer (JavaScript, Python, LLM)",Bloomentum,16
869,itviec,itviec_sample_50_clean.csv,https://itviec.com/it-jobs/ai-automation-engineer-javascript-python-llm-bloomentum-2901,"AI Automation Engineer (JavaScript, Python, LLM)",Bloomentum,16


Rows after URL dedupe: 1,433


,source,rows,numeric_salary_rows,salary_raw_non_empty,skills_non_empty,experience_non_empty,salary_numeric_fill_rate
0,itviec,750,207,750,750,658,0.276000
1,topcv,583,305,534,454,495,0.523156
2,topdev,100,0,100,100,83,0.000000


## TopDev coverage

TopDev is useful for inventory, demand, skills, location, and experience checks. It should not contribute to salary numeric summaries unless future crawls expose numeric salaries.

In [7]:
topdev = jobs_deduped[jobs_deduped["source"].eq("topdev")].copy()
print(f"TopDev deduped rows: {len(topdev):,}")
print(f"TopDev numeric salary rows: {int(topdev['_has_numeric_salary'].sum()):,}")

display(topdev["salary_raw"].fillna("<NA>").astype(str).str.strip().replace("", "<empty>").value_counts().rename_axis("salary_raw").reset_index(name="rows"))
display(topdev[["title", "company", "location", "salary_raw", "experience_raw", "skills", "url"]].head(20))

TopDev deduped rows: 100
TopDev numeric salary rows: 0


,salary_raw,rows
0,Login to view salary,100


,title,company,location,salary_raw,experience_raw,skills,url
1649,"[2026_SVXS_CNTT-DL]_Sinh viên tài năng Khối Công nghệ thông tin, Dữ liệu",Vietcombank,Hà Nội,Login to view salary,fresher,"Computer Science, Data Engineer, AI",https://topdev.vn/detail-jobs/2026-svxs-cntt-dl-sinh-vien-tai-nang-khoi-cong-nghe-thong-tin-du-lieu-vietcombank-2111848
1573,Agentic Engineer (Python/Go/C/C++ Backend & AI Agents) - Khối Công nghệ thông tin (2026TD450887),MBBANK,Hà Nội,Login to view salary,fresher,"Python, Go, C/C++, AI, PostgreSQL, Tensorflow, Computer Vision, Docker, Kubernetes, MongoDB",https://topdev.vn/detail-jobs/agentic-engineer-python-go-c-c-backend-ai-agents-khoi-cong-nghe-thong-tin-2026td450887...
1575,AI Agent Engineer (Middle/Senior),"CyberLogitec Vietnam Co., Ltd.",Hồ Chí Minh,Login to view salary,5+ years,"Python, REST API, AI, React, FastAPI, Vue, PostgreSQL, Redis",https://topdev.vn/detail-jobs/ai-agent-engineer-middle-senior-cyberlogitec-vietnam-co-ltd-2115298
1567,AI & DATA ENGINEER,CÔNG TY TNHH DREAM TALENT,Hồ Chí Minh,Login to view salary,Minimum 3 years,"SQL, Python, Data Engineer, AWS, Azure, GCP",https://topdev.vn/detail-jobs/ai-data-engineer-cong-ty-tnhh-dream-talent-2115438
1578,AI Engineer,AETOSKY PTE. LTD,Hà Nội,Login to view salary,4–8 years,"Python, Git, Docker, Database, Machine Learning, API, AI",https://topdev.vn/detail-jobs/ai-engineer-aetosky-pte-ltd-2115966
1591,AI Engineer (Computer Vision/ NLP/ LLM) - Khối Công Nghệ Thông Tin (2026TD450582),MBBANK,Hà Nội,Login to view salary,NaN,"Python, Machine Learning, C/C++, AI, Deep Learning, Tensorflow, FastAPI, Docker",https://topdev.vn/detail-jobs/ai-engineer-computer-vision-nlp-llm-khoi-cong-nghe-thong-tin-2026td450582-mbbank-2113056
1565,AI ENGINEER,CÔNG TY TNHH PHẦN MỀM NAM LONG,Hồ Chí Minh,Login to view salary,2+ năm,"Python, API, AI",https://topdev.vn/detail-jobs/ai-engineer-cong-ty-tnhh-phan-mem-nam-long-2114873
1571,AI FULLSTACK DEVELOPER,CÔNG TY TNHH SOCOTEC VIỆT NAM,Hà Nội,Login to view salary,NaN,"Python, AI, React",https://topdev.vn/detail-jobs/ai-fullstack-developer-cong-ty-tnhh-socotec-viet-nam-2110330
1574,AI Software Engineer (Python/Go/C/C++ Backend & AI Agents) - Khối Công nghệ thông tin (2026TD450888),MBBANK,Hà Nội,Login to view salary,fresher,"Python, Go, C/C++, AI, PostgreSQL, Tensorflow, Computer Vision, Docker, Kubernetes, MongoDB",https://topdev.vn/detail-jobs/ai-software-engineer-python-go-c-c-backend-ai-agents-khoi-cong-nghe-thong-tin-2026td45...
1612,Backend Developer (Java),NGÂN HÀNG THƯƠNG MẠI CỔ PHẦN SÀI GÒN TÀI LỘC (SACOMBANK),Hồ Chí Minh,Login to view salary,NaN,"OOP, Oracle, Java, Restful Api, Spring Boot, Docker, Kubernetes, Redis, DevOps, iOS, Android",https://topdev.vn/detail-jobs/backend-developer-java-ngan-hang-thuong-mai-co-phan-sai-gon-tai-loc-sacombank-2110332


## Salary EDA

Salary summaries use the deduped view and only rows with at least one numeric salary bound. Currencies and periods are not mixed.

In [8]:
salary_rows = jobs_deduped[jobs_deduped["_has_numeric_salary"]].copy()


def salary_suspicious_reason(row):
    reasons = []
    if pd.notna(row.get("salary_min")) and row["salary_min"] <= 0:
        reasons.append("salary_min_non_positive")
    if pd.notna(row.get("salary_min")) and pd.notna(row.get("salary_max")) and row["salary_min"] > row["salary_max"]:
        reasons.append("salary_min_gt_max")
    if pd.isna(row.get("salary_currency")) or str(row.get("salary_currency", "")).strip() == "":
        reasons.append("numeric_salary_missing_currency")
    if str(row.get("salary_currency", "")).upper() == "VND" and pd.notna(row.get("_salary_midpoint")) and row["_salary_midpoint"] > 500_000_000:
        reasons.append("vnd_midpoint_gt_500m")
    return ";".join(reasons)


salary_rows["salary_suspicious_reason"] = salary_rows.apply(salary_suspicious_reason, axis=1)
salary_rows_clean = salary_rows[salary_rows["salary_suspicious_reason"].eq("")].copy()
suspicious_salary = salary_rows[salary_rows["salary_suspicious_reason"].ne("")].copy()

salary_coverage = (
    jobs_deduped.groupby("source", dropna=False)
    .agg(rows=("url", "size"), numeric_salary_rows=("_has_numeric_salary", "sum"))
    .reset_index()
)
salary_coverage["numeric_salary_rate"] = salary_coverage["numeric_salary_rows"] / salary_coverage["rows"]
display(salary_coverage.sort_values("source"))

salary_summary = (
    salary_rows_clean.groupby(["source", "salary_currency", "salary_period"], dropna=False)
    .agg(
        salary_count=("url", "nunique"),
        midpoint_median=("_salary_midpoint", "median"),
        midpoint_min=("_salary_midpoint", "min"),
        midpoint_max=("_salary_midpoint", "max"),
        salary_min_median=("salary_min", "median"),
        salary_max_median=("salary_max", "median"),
    )
    .reset_index()
    .sort_values(["source", "salary_currency", "salary_period"])
)
display(salary_summary)

print(f"Suspicious salary rows: {len(suspicious_salary):,}")
display(suspicious_salary[["salary_suspicious_reason", "source", "title", "company", "location", "salary_raw", "salary_min", "salary_max", "salary_currency", "salary_period", "url"]])

,source,rows,numeric_salary_rows,numeric_salary_rate
0,itviec,750,207,0.276000
1,topcv,583,305,0.523156
2,topdev,100,0,0.000000


,source,salary_currency,salary_period,salary_count,midpoint_median,midpoint_min,midpoint_max,salary_min_median,salary_max_median
0,itviec,USD,month,133,1500.0,230.0,7000.0,1000.0,2000.0
1,itviec,USD,year,43,1500.0,300.0,3250.0,1000.0,2000.0
2,itviec,VND,month,24,34250000.0,16500000.0,95000000.0,30000000.0,35000000.0
3,itviec,VND,year,2,33750000.0,17500000.0,50000000.0,15000000.0,35000000.0
4,topcv,USD,month,6,1075.0,600.0,1500.0,650.0,1500.0
5,topcv,VND,month,240,17500000.0,1750000.0,83500000.0,14000000.0,23000000.0
6,topcv,VND,year,56,18750000.0,2500000.0,57500000.0,14500000.0,25000000.0


Suspicious salary rows: 8


,salary_suspicious_reason,source,title,company,location,salary_raw,salary_min,salary_max,salary_currency,salary_period,url
872,salary_min_non_positive,itviec,C++ Developer (JP N3+),FPT Software,Hồ Chí Minh,0 - 500 USD,0.0,500.0,USD,year,https://itviec.com/it-jobs/c-developer-jp-n3-fpt-software-0354
413,salary_min_non_positive,itviec,"Frontend Developer Intern (HTML, CSS, JavaScript)",EZ Games,Hồ Chí Minh,0 - 500 USD,0.0,500.0,USD,month,https://itviec.com/it-jobs/frontend-developer-intern-html-css-javascript-ez-games-0012
609,salary_min_non_positive,itviec,QA/ Manual Tester Intern (QA QC),SRT GROUP,Hồ Chí Minh,"0 - 1,000 USD",0.0,1000.0,USD,year,https://itviec.com/it-jobs/qa-manual-tester-intern-qa-qc-srt-group-2633
227,numeric_salary_missing_currency,itviec,Senior/Expert Data Analyst (5YOE - Banking Finance),F88,Hà Nội,"30,000,000-45,000,000",30000000.0,45000000.0,NaN,year,https://itviec.com/it-jobs/senior-expert-data-analyst-5yoe-banking-finance-f88-2837
452,numeric_salary_missing_currency,itviec,"Sr .NET Backend Developer (ASP.NET, C#, SQL, ReactJS)",GrapeCity,Hà Nội,"30,000,000 - 50,000,000đ",30000000.0,50000000.0,NaN,month,https://itviec.com/it-jobs/sr-net-backend-developer-asp-net-c-sql-reactjs-grapecity-3733
967,numeric_salary_missing_currency,topcv,Fullstack Developer (From 3 YoE) - Domain E-Commerce Product,XIPAT FLEXIBLE SOLUTIONS COMPANY LIMITED,Ha Noi,From 3,3.0,NaN,NaN,year,https://www.topcv.vn/viec-lam/fullstack-developer-from-3-yoe-domain-e-commerce-product/2204957.html
1529,vnd_midpoint_gt_500m,topcv,Kỹ Sư Phần Mềm Hệ Thống Điều Khiển (C#),CÔNG TY TNHH VTI EDUCATION,Nhật Bản,860000000 - 870000000 VND,860000000.0,870000000.0,VND,year,https://www.topcv.vn/viec-lam/ky-su-phan-mem-he-thong-dieu-khien-c/2000152.html
1067,numeric_salary_missing_currency,topcv,Senior Business Analyst (Domain Retail) - Offer Upto 1500$,Công ty Cổ phần MISA,Ha Noi,Upto 1500,NaN,1500.0,NaN,month,https://www.topcv.vn/viec-lam/senior-business-analyst-domain-retail-offer-upto-1500/2220935.html


## Skills and demand exploration

Demand uses all deduped sources, including TopDev. Salary by skill uses clean numeric salary rows only.

In [9]:
def parse_listish(value):
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text:
        return []
    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                return [str(item).strip() for item in parsed if str(item).strip()]
        except (ValueError, SyntaxError):
            pass
    return [part.strip() for part in text.replace(";", ",").split(",") if part.strip()]


skill_rows = jobs_deduped.assign(skill=jobs_deduped["skills"].apply(parse_listish)).explode("skill")
skill_rows = skill_rows[skill_rows["skill"].fillna("").astype(str).str.strip().ne("")].copy()

skill_coverage = (
    jobs_deduped.groupby("source", dropna=False)
    .agg(rows=("url", "size"), rows_with_skills=("_has_skills", "sum"))
    .reset_index()
)
skill_coverage["skills_fill_rate"] = skill_coverage["rows_with_skills"] / skill_coverage["rows"]
display(skill_coverage.sort_values("source"))

demand_by_skill = (
    skill_rows.groupby(["source", "skill"], dropna=False)
    .agg(job_count=("url", "nunique"))
    .reset_index()
    .sort_values(["job_count", "source", "skill"], ascending=[False, True, True])
)
display(demand_by_skill.head(40))

salary_skill_rows = salary_rows_clean.assign(skill=salary_rows_clean["skills"].apply(parse_listish)).explode("skill")
salary_skill_rows = salary_skill_rows[salary_skill_rows["skill"].fillna("").astype(str).str.strip().ne("")].copy()
salary_by_skill = (
    salary_skill_rows.groupby(["source", "skill", "salary_currency", "salary_period"], dropna=False)
    .agg(
        salary_count=("url", "nunique"),
        midpoint_median=("_salary_midpoint", "median"),
        midpoint_min=("_salary_midpoint", "min"),
        midpoint_max=("_salary_midpoint", "max"),
    )
    .reset_index()
)
display(salary_by_skill[salary_by_skill["salary_count"].ge(2)].sort_values(["salary_count", "source", "skill"], ascending=[False, True, True]).head(40))

,source,rows,rows_with_skills,skills_fill_rate
0,itviec,750,750,1.000000
1,topcv,583,454,0.778731
2,topdev,100,100,1.000000


,source,skill,job_count
251,itviec,SQL,252
228,itviec,Python,242
10,itviec,AWS,218
88,itviec,Docker,183
152,itviec,Java,180
100,itviec,English,168
5,itviec,AI,159
84,itviec,DevOps,156
29,itviec,Azure,146
163,itviec,Kubernetes,134


,source,skill,salary_currency,salary_period,salary_count,midpoint_median,midpoint_min,midpoint_max
312,itviec,SQL,USD,month,50,1525.0,400.0,3300.0
280,itviec,Python,USD,month,42,1500.0,300.0,3250.0
14,itviec,AWS,USD,month,36,1650.0,650.0,7000.0
1068,topcv,SQL,VND,month,35,20000000.0,4500000.0,55000000.0
6,itviec,AI,USD,month,29,1600.0,230.0,3250.0
103,itviec,Docker,USD,month,29,1500.0,950.0,3300.0
179,itviec,JavaScript,USD,month,27,1500.0,350.0,4000.0
97,itviec,DevOps,USD,month,26,1450.0,350.0,3300.0
116,itviec,English,USD,month,23,1650.0,900.0,3500.0
176,itviec,Java,USD,month,23,1500.0,350.0,3500.0


## Current-data checks

These checks document the current repository state. If new processed files are added later, rerun the notebook and update the expectations intentionally.

In [10]:
current_checks = {
    "clean_csv_files": len(clean_paths),
    "all_rows_loaded": len(jobs),
    "unique_urls_all_rows": jobs["url"].nunique(),
    "deduped_rows": len(jobs_deduped),
    "parse_status_values": sorted(jobs["parse_status"].dropna().astype(str).unique().tolist()),
    "topdev_numeric_salary_rows": int(jobs_deduped.loc[jobs_deduped["source"].eq("topdev"), "_has_numeric_salary"].sum()),
    "numeric_salary_rows_deduped": int(jobs_deduped["_has_numeric_salary"].sum()),
    "suspicious_salary_rows": len(suspicious_salary),
}
display(pd.DataFrame([current_checks]).T.rename(columns={0: "value"}))

,value
clean_csv_files,8
all_rows_loaded,1662
unique_urls_all_rows,1433
deduped_rows,1433
parse_status_values,[ok]
topdev_numeric_salary_rows,0
numeric_salary_rows_deduped,512
suspicious_salary_rows,8


## Analysis Cleaning Pipeline

Ph?n n?y t?o m?t nh?nh d? li?u ph?n t?ch s?ch h?n t? `jobs` ?? load ? tr?n. Ta gi? `jobs` l?m d? li?u EDA g?c, c?n `jobs_clean` l? b?n ?? chu?n h?a ?? ph?n t?ch.

Pandas c?n ch? ?: `.copy()` ?? t?ch nh?nh d? li?u, `.loc` ?? filter an to?n, `.astype("string")` v? `.str` ?? l?m s?ch text, `pd.to_numeric`/`pd.to_datetime` ?? ?p ki?u c? ki?m so?t.


In [11]:
import re
import unicodedata

ANALYSIS_DIR = REPO_ROOT / "data" / "analysis"

analysis_text_columns = [
    "source",
    "url",
    "job_id",
    "title",
    "company",
    "location",
    "salary_raw",
    "salary_currency",
    "skills",
    "experience_raw",
    "description",
    "parse_status",
    "posted_raw",
    "location_cities",
    "seniority",
    "work_mode",
    "employment_type",
    "salary_period",
]
analysis_numeric_columns = ["salary_min", "salary_max", "experience_min", "experience_max"]
analysis_date_columns = ["scraped_at", "posted_at", "valid_through"]


def fold_text(value):
    """Fold text for matching without changing the display columns."""
    if pd.isna(value):
        return ""
    text = str(value)
    text = "".join(
        character
        for character in unicodedata.normalize("NFD", text)
        if unicodedata.category(character) != "Mn"
    )
    return re.sub(r"\s+", " ", text).strip().casefold()


def clean_text_series(series):
    cleaned = series.astype("string").str.replace(r"\s+", " ", regex=True).str.strip()
    return cleaned.mask(cleaned.eq(""), pd.NA)


jobs_clean = jobs.copy()
input_rows = len(jobs_clean)

for column in analysis_text_columns:
    if column in jobs_clean.columns:
        jobs_clean[column] = clean_text_series(jobs_clean[column])

for column in analysis_numeric_columns:
    if column in jobs_clean.columns:
        jobs_clean[column] = pd.to_numeric(jobs_clean[column], errors="coerce")

for column in analysis_date_columns:
    if column in jobs_clean.columns:
        jobs_clean[f"{column}_parsed"] = pd.to_datetime(jobs_clean[column], errors="coerce", utc=True, format="mixed")

jobs_clean = jobs_clean.loc[jobs_clean["parse_status"].eq("ok")].copy()

cleaning_start_audit = pd.DataFrame(
    [
        {"metric": "input_rows", "value": input_rows},
        {"metric": "parse_ok_rows", "value": len(jobs_clean)},
        {"metric": "unique_urls_before_dedupe", "value": jobs_clean["url"].nunique()},
        {"metric": "duplicate_url_rows_before_dedupe", "value": int(jobs_clean["url"].duplicated(keep=False).sum())},
    ]
)
display(cleaning_start_audit)


,metric,value
0,input_rows,1662
1,parse_ok_rows,1662
2,unique_urls_before_dedupe,1433
3,duplicate_url_rows_before_dedupe,421


## 1. Dedupe theo URL

M?t job c? th? xu?t hi?n ? nhi?u run. Ta sort ?? b?n gi? l?i l? b?n c? nhi?u th?ng tin nh?t: `_quality_score` cao h?n, `scraped_at` m?i h?n, r?i `_input_file` l?m fallback ?n ??nh.

Pandas c?n ch? ?: `.duplicated()` ?? audit duplicate, `.sort_values()` ?? ??nh ngh?a ?u ti?n, `.drop_duplicates(..., keep="last")` ?? gi? b?n t?t nh?t sau khi sort.


In [12]:
jobs_clean["_has_numeric_salary"] = jobs_clean[["salary_min", "salary_max"]].notna().any(axis=1)
jobs_clean["_salary_midpoint"] = jobs_clean[["salary_min", "salary_max"]].mean(axis=1)
jobs_clean["_has_skills"] = jobs_clean["skills"].notna()
jobs_clean["_has_experience"] = jobs_clean["experience_raw"].notna()
jobs_clean["_has_description"] = jobs_clean["description"].notna()
jobs_clean["_quality_score"] = (
    jobs_clean["_has_numeric_salary"].astype(int) * 100
    + jobs_clean["_has_skills"].astype(int) * 10
    + jobs_clean["_has_experience"].astype(int) * 5
    + jobs_clean["_has_description"].astype(int)
)
jobs_clean["_scraped_at_sort"] = jobs_clean["scraped_at_parsed"].fillna(pd.Timestamp("1900-01-01", tz="UTC"))

before_dedupe_rows = len(jobs_clean)
duplicate_rows = jobs_clean.loc[jobs_clean["url"].duplicated(keep=False)].copy()

jobs_clean = (
    jobs_clean.sort_values(["url", "_quality_score", "_scraped_at_sort", "_input_file"], na_position="first")
    .drop_duplicates("url", keep="last")
    .reset_index(drop=True)
)

dedupe_audit = pd.DataFrame(
    [
        {"metric": "rows_before_dedupe", "value": before_dedupe_rows},
        {"metric": "rows_after_dedupe", "value": len(jobs_clean)},
        {"metric": "rows_removed", "value": before_dedupe_rows - len(jobs_clean)},
        {"metric": "unique_urls_after_dedupe", "value": jobs_clean["url"].nunique()},
    ]
)
display(dedupe_audit)

display(
    duplicate_rows[["source", "_input_file", "url", "title", "company", "_quality_score", "scraped_at_parsed"]]
    .sort_values(["url", "_quality_score", "scraped_at_parsed"], na_position="first")
    .head(30)
)


,metric,value
0,rows_before_dedupe,1662
1,rows_after_dedupe,1433
2,rows_removed,229
3,unique_urls_after_dedupe,1433


,source,_input_file,url,title,company,_quality_score,scraped_at_parsed
863,itviec,itviec_sample_50_clean.csv,https://itviec.com/it-jobs/agentic-engineer-python-go-c-c-backend-ai-agents-mb-bank-1315,Agentic Engineer (Python/Go/C/C++ Backend & AI Agents),MB Bank,16,2026-06-30 09:03:48.858747+00:00
12,itviec,itviec_20260701_100_clean.csv,https://itviec.com/it-jobs/agentic-engineer-python-go-c-c-backend-ai-agents-mb-bank-1315,Agentic Engineer (Python/Go/C/C++ Backend & AI Agents),MB Bank,16,2026-06-30 17:21:46.300901+00:00
112,itviec,itviec_salary_20260702_clean.csv,https://itviec.com/it-jobs/agentic-engineer-python-go-c-c-backend-ai-agents-mb-bank-1315,Agentic Engineer (Python/Go/C/C++ Backend & AI Agents),MB Bank,16,2026-06-30 17:21:46.300901+00:00
49,itviec,itviec_20260701_100_clean.csv,https://itviec.com/it-jobs/ai-agent-engineer-ml-python-typesript-net-adk-truong-minh-thinh-technology-joint-stock-co...,"AI Agent Engineer (ML, Python, TypeSript, .NET, ADK)",TRUONG MINH THINH TECHNOLOGY JOINT STOCK COMPANY,16,2026-06-30 17:28:14.851088+00:00
149,itviec,itviec_salary_20260702_clean.csv,https://itviec.com/it-jobs/ai-agent-engineer-ml-python-typesript-net-adk-truong-minh-thinh-technology-joint-stock-co...,"AI Agent Engineer (ML, Python, TypeSript, .NET, ADK)",TRUONG MINH THINH TECHNOLOGY JOINT STOCK COMPANY,16,2026-06-30 17:28:14.851088+00:00
28,itviec,itviec_20260701_100_clean.csv,https://itviec.com/it-jobs/ai-agent-engineer-python-llm-kds-vietnam-5513,"AI Agent Engineer (Python, LLM)",KDS Vietnam,116,2026-06-30 17:22:38.298026+00:00
128,itviec,itviec_salary_20260702_clean.csv,https://itviec.com/it-jobs/ai-agent-engineer-python-llm-kds-vietnam-5513,"AI Agent Engineer (Python, LLM)",KDS Vietnam,116,2026-06-30 17:22:38.298026+00:00
869,itviec,itviec_sample_50_clean.csv,https://itviec.com/it-jobs/ai-automation-engineer-javascript-python-llm-bloomentum-2901,"AI Automation Engineer (JavaScript, Python, LLM)",Bloomentum,16,2026-06-30 09:04:10.501743+00:00
16,itviec,itviec_20260701_100_clean.csv,https://itviec.com/it-jobs/ai-automation-engineer-javascript-python-llm-bloomentum-2901,"AI Automation Engineer (JavaScript, Python, LLM)",Bloomentum,16,2026-06-30 17:22:01.237787+00:00
116,itviec,itviec_salary_20260702_clean.csv,https://itviec.com/it-jobs/ai-automation-engineer-javascript-python-llm-bloomentum-2901,"AI Automation Engineer (JavaScript, Python, LLM)",Bloomentum,16,2026-06-30 17:22:01.237787+00:00


## 2. L?m s?ch salary ?? ph?n t?ch

Ta kh?ng c? bi?n m?i `salary_raw` th?nh s?. V?i d? li?u job, c?c label nh? `Negotiable`, `Tho? thu?n`, `Login to view salary`, `You'll love it` l? t?n hi?u ri?ng v? c?n gi? l?i.

Pandas c?n ch? ?: t?o mask boolean b?ng `.notna()`, `.str.contains()`, c?p nh?t c? ?i?u ki?n b?ng `.loc`, v? d?ng `.groupby().size()` ?? ki?m tra ph?n b? label sau cleaning.


In [13]:
jobs_clean["salary_midpoint"] = jobs_clean[["salary_min", "salary_max"]].mean(axis=1)
jobs_clean["salary_suspicious_reason"] = ""
jobs_clean["_salary_raw_folded"] = jobs_clean["salary_raw"].map(fold_text)

numeric_salary_mask = jobs_clean[["salary_min", "salary_max"]].notna().any(axis=1)


def append_salary_reason(mask, reason):
    mask = mask.fillna(False)
    if not mask.any():
        return
    existing = jobs_clean.loc[mask, "salary_suspicious_reason"].astype("string")
    jobs_clean.loc[mask, "salary_suspicious_reason"] = existing.where(existing.eq(""), existing + ";") + reason


append_salary_reason(
    numeric_salary_mask & jobs_clean["salary_min"].notna() & jobs_clean["salary_min"].le(0),
    "salary_min_non_positive",
)
append_salary_reason(
    numeric_salary_mask
    & jobs_clean["salary_min"].notna()
    & jobs_clean["salary_max"].notna()
    & jobs_clean["salary_min"].gt(jobs_clean["salary_max"]),
    "salary_min_gt_max",
)
append_salary_reason(
    numeric_salary_mask & jobs_clean["salary_currency"].isna(),
    "numeric_salary_missing_currency",
)
append_salary_reason(
    numeric_salary_mask
    & jobs_clean["salary_currency"].fillna("").astype("string").str.upper().eq("VND")
    & jobs_clean["salary_midpoint"].gt(500_000_000),
    "vnd_midpoint_gt_500m",
)

hidden_salary_mask = jobs_clean["_salary_raw_folded"].str.contains(r"login|log in|view salary|hidden", na=False)
negotiable_salary_mask = jobs_clean["_salary_raw_folded"].str.contains(
    r"negotiable|negotiation|thoa thuan|thuong luong|deal|discuss", na=False
)
salary_raw_present = jobs_clean["salary_raw"].notna()

jobs_clean["salary_label"] = "missing"
jobs_clean.loc[numeric_salary_mask, "salary_label"] = "numeric"
jobs_clean.loc[numeric_salary_mask & jobs_clean["salary_suspicious_reason"].ne(""), "salary_label"] = "numeric_suspicious"
jobs_clean.loc[~numeric_salary_mask & hidden_salary_mask, "salary_label"] = "hidden"
jobs_clean.loc[~numeric_salary_mask & negotiable_salary_mask, "salary_label"] = "negotiable"
jobs_clean.loc[
    ~numeric_salary_mask & salary_raw_present & ~hidden_salary_mask & ~negotiable_salary_mask,
    "salary_label",
] = "non_numeric_label"

salary_clean = jobs_clean.loc[jobs_clean["salary_label"].eq("numeric")].copy()
suspicious_salary_cleaning = jobs_clean.loc[jobs_clean["salary_label"].eq("numeric_suspicious")].copy()

salary_label_summary = (
    jobs_clean.groupby(["source", "salary_label"], dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values(["source", "salary_label"])
)
display(salary_label_summary)

salary_numeric_summary = (
    salary_clean.groupby(["source", "salary_currency", "salary_period"], dropna=False)
    .agg(
        jobs=("url", "nunique"),
        midpoint_median=("salary_midpoint", "median"),
        midpoint_min=("salary_midpoint", "min"),
        midpoint_max=("salary_midpoint", "max"),
    )
    .reset_index()
    .sort_values(["source", "salary_currency", "salary_period"])
)
display(salary_numeric_summary)

display(
    suspicious_salary_cleaning[
        [
            "salary_suspicious_reason",
            "source",
            "title",
            "company",
            "salary_raw",
            "salary_min",
            "salary_max",
            "salary_currency",
            "url",
        ]
    ].sort_values(["source", "salary_suspicious_reason", "title"])
)


,source,salary_label,rows
0,itviec,negotiable,12
1,itviec,non_numeric_label,531
2,itviec,numeric,202
3,itviec,numeric_suspicious,5
4,topcv,missing,49
5,topcv,negotiable,229
6,topcv,numeric,302
7,topcv,numeric_suspicious,3
8,topdev,hidden,100


,source,salary_currency,salary_period,jobs,midpoint_median,midpoint_min,midpoint_max
0,itviec,USD,month,133,1500.0,230.0,7000.0
1,itviec,USD,year,43,1500.0,300.0,3250.0
2,itviec,VND,month,24,34250000.0,16500000.0,95000000.0
3,itviec,VND,year,2,33750000.0,17500000.0,50000000.0
4,topcv,USD,month,6,1075.0,600.0,1500.0
5,topcv,VND,month,240,17500000.0,1750000.0,83500000.0
6,topcv,VND,year,56,18750000.0,2500000.0,57500000.0


,salary_suspicious_reason,source,title,company,salary_raw,salary_min,salary_max,salary_currency,url
561,numeric_salary_missing_currency,itviec,Senior/Expert Data Analyst (5YOE - Banking Finance),F88,"30,000,000-45,000,000",30000000.0,45000000.0,<NA>,https://itviec.com/it-jobs/senior-expert-data-analyst-5yoe-banking-finance-f88-2837
701,numeric_salary_missing_currency,itviec,"Sr .NET Backend Developer (ASP.NET, C#, SQL, ReactJS)",GrapeCity,"30,000,000 - 50,000,000đ",30000000.0,50000000.0,<NA>,https://itviec.com/it-jobs/sr-net-backend-developer-asp-net-c-sql-reactjs-grapecity-3733
125,salary_min_non_positive,itviec,C++ Developer (JP N3+),FPT Software,0 - 500 USD,0.0,500.0,USD,https://itviec.com/it-jobs/c-developer-jp-n3-fpt-software-0354
217,salary_min_non_positive,itviec,"Frontend Developer Intern (HTML, CSS, JavaScript)",EZ Games,0 - 500 USD,0.0,500.0,USD,https://itviec.com/it-jobs/frontend-developer-intern-html-css-javascript-ez-games-0012
459,salary_min_non_positive,itviec,QA/ Manual Tester Intern (QA QC),SRT GROUP,"0 - 1,000 USD",0.0,1000.0,USD,https://itviec.com/it-jobs/qa-manual-tester-intern-qa-qc-srt-group-2633
1008,numeric_salary_missing_currency,topcv,Fullstack Developer (From 3 YoE) - Domain E-Commerce Product,XIPAT FLEXIBLE SOLUTIONS COMPANY LIMITED,From 3,3.0,NaN,<NA>,https://www.topcv.vn/viec-lam/fullstack-developer-from-3-yoe-domain-e-commerce-product/2204957.html
1318,numeric_salary_missing_currency,topcv,Senior Business Analyst (Domain Retail) - Offer Upto 1500$,Công ty Cổ phần MISA,Upto 1500,NaN,1500.0,<NA>,https://www.topcv.vn/viec-lam/senior-business-analyst-domain-retail-offer-upto-1500/2220935.html
1103,vnd_midpoint_gt_500m,topcv,Kỹ Sư Phần Mềm Hệ Thống Điều Khiển (C#),CÔNG TY TNHH VTI EDUCATION,860000000 - 870000000 VND,860000000.0,870000000.0,VND,https://www.topcv.vn/viec-lam/ky-su-phan-mem-he-thong-dieu-khien-c/2000152.html


## 3. Chu?n h?a skills v? cities

C?c c?t list khi ??c t? CSV th??ng th?nh text. Ta chuy?n ch?ng v? list th?t, r?i d?ng `.explode()` ?? t?o b?ng ph?: m?t d?ng l? m?t job-skill ho?c m?t job-city.

Pandas c?n ch? ?: `.apply()` ?? parse t?ng cell listish, `.explode()` ?? bung list th?nh nhi?u d?ng, `.drop_duplicates()` ?? tr?nh ??m tr?ng skill/city trong c?ng m?t job.


In [14]:
def to_listish(value):
    if isinstance(value, list):
        return [str(item).strip() for item in value if str(item).strip()]
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text:
        return []
    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                return [str(item).strip() for item in parsed if str(item).strip()]
        except (ValueError, SyntaxError):
            pass
    return [part.strip() for part in text.replace(";", ",").split(",") if part.strip()]


skill_alias = {
    "js": "javascript",
    "reactjs": "react",
    "react.js": "react",
    "vuejs": "vue",
    "vue.js": "vue",
    "node": "node.js",
    "nodejs": "node.js",
    "golang": "go",
    "py": "python",
    "postgres": "postgresql",
    "k8s": "kubernetes",
    "ts": "typescript",
    "csharp": "c#",
    "dotnet": ".net",
    "ci cd": "ci/cd",
    "html5": "html",
    "css3": "css",
    "nextjs": "next.js",
    "nestjs": "nest.js",
}


def normalize_skill(value):
    folded = fold_text(value)
    folded = folded.replace("&", " and ")
    folded = re.sub(r"\s+", " ", folded).strip()
    return skill_alias.get(folded, folded)


city_alias = {
    "hcm": "Ho Chi Minh",
    "ho chi minh": "Ho Chi Minh",
    "thanh pho ho chi minh": "Ho Chi Minh",
    "sai gon": "Ho Chi Minh",
    "ha noi": "Ha Noi",
    "hanoi": "Ha Noi",
    "da nang": "Da Nang",
    "danang": "Da Nang",
    "binh duong": "Binh Duong",
    "dong nai": "Dong Nai",
    "can tho": "Can Tho",
    "hai phong": "Hai Phong",
}


def normalize_city(value):
    folded = fold_text(value)
    if not folded:
        return ""
    return city_alias.get(folded, str(value).strip())


jobs_clean["skills_list"] = jobs_clean["skills"].apply(to_listish)
jobs_clean["cities_list"] = jobs_clean["location_cities"].apply(to_listish)

job_skills = (
    jobs_clean[["url", "source", "title", "company", "skills_list"]]
    .explode("skills_list", ignore_index=True)
    .rename(columns={"skills_list": "skill"})
)
job_skills["skill"] = clean_text_series(job_skills["skill"])
job_skills["skill_norm"] = job_skills["skill"].map(normalize_skill).astype("string")
job_skills = (
    job_skills.loc[job_skills["skill_norm"].notna() & job_skills["skill_norm"].ne("")]
    .drop_duplicates(["url", "skill_norm"])
    .sort_values(["source", "skill_norm", "url"])
    .reset_index(drop=True)
)

skill_norm_lookup = job_skills.groupby("url")["skill_norm"].agg(list)
jobs_clean["skills_norm_list"] = jobs_clean["url"].map(skill_norm_lookup).apply(lambda value: value if isinstance(value, list) else [])
jobs_clean["skill_count"] = jobs_clean["skills_norm_list"].str.len()

job_cities = (
    jobs_clean[["url", "source", "title", "company", "location", "cities_list"]]
    .explode("cities_list", ignore_index=True)
    .rename(columns={"cities_list": "city"})
)
job_cities["city"] = clean_text_series(job_cities["city"]).map(normalize_city).astype("string")
job_cities = (
    job_cities.loc[job_cities["city"].notna() & job_cities["city"].ne("")]
    .drop_duplicates(["url", "city"])
    .sort_values(["source", "city", "url"])
    .reset_index(drop=True)
)

city_lookup = job_cities.groupby("url")["city"].agg(list)
jobs_clean["cities_list"] = jobs_clean["url"].map(city_lookup).apply(lambda value: value if isinstance(value, list) else [])
jobs_clean["city_count"] = jobs_clean["cities_list"].str.len()

skill_demand = job_skills.groupby(["source", "skill_norm"], dropna=False).agg(jobs=("url", "nunique")).reset_index()
city_demand = job_cities.groupby(["source", "city"], dropna=False).agg(jobs=("url", "nunique")).reset_index()

display(skill_demand.sort_values(["jobs", "source", "skill_norm"], ascending=[False, True, True]).head(40))
display(city_demand.sort_values(["source", "jobs", "city"], ascending=[True, False, True]).head(40))


,source,skill_norm,jobs
270,itviec,sql,252
228,itviec,python,242
25,itviec,aws,218
88,itviec,docker,183
149,itviec,java,180
98,itviec,english,168
8,itviec,ai,159
84,itviec,devops,156
29,itviec,azure,146
164,itviec,kubernetes,134


,source,city,jobs
2,itviec,Da Nang,750
3,itviec,Ha Noi,750
5,itviec,Ho Chi Minh,750
1,itviec,Can Tho,38
0,itviec,Binh Duong,2
4,itviec,Hai Phong,2
6,topcv,Binh Duong,583
7,topcv,Can Tho,583
8,topcv,Da Nang,583
9,topcv,Dong Nai,583


## 4. Experience buckets v? date fields

Ta gi? `experience_min/max` d?ng s?, r?i t?o bucket ?? ph?n t?ch ph?n b? seniority d? h?n. V?i ng?y th?ng, ta t?o `posted_week` ?? ph?c v? trend theo tu?n.

Pandas c?n ch? ?: `pd.cut()` ?? t?o bucket, `pd.to_datetime(..., format="mixed")` ?? x? l? format ng?y b? tr?n, `.dt.isocalendar()` ?? l?y ISO week, `.dt.strftime()` ?? t?o chu?i ng?y d? export.


In [15]:
jobs_clean["experience_bucket"] = pd.cut(
    jobs_clean["experience_min"],
    bins=[-1, 0, 1, 3, 5, 10, 100],
    labels=["0", "0-1", "1-3", "3-5", "5-10", "10+"],
)
jobs_clean["experience_bucket"] = jobs_clean["experience_bucket"].astype("string").fillna("unknown")

posted_iso = jobs_clean["posted_at_parsed"].dt.isocalendar()
jobs_clean["posted_week"] = posted_iso["year"].astype("string") + "-W" + posted_iso["week"].astype("string").str.zfill(2)
jobs_clean.loc[jobs_clean["posted_at_parsed"].isna(), "posted_week"] = pd.NA
jobs_clean["posted_date"] = jobs_clean["posted_at_parsed"].dt.strftime("%Y-%m-%d")
jobs_clean["scraped_date"] = jobs_clean["scraped_at_parsed"].dt.strftime("%Y-%m-%d")

experience_anomalies = jobs_clean.loc[
    jobs_clean["experience_min"].notna()
    & jobs_clean["experience_max"].notna()
    & jobs_clean["experience_min"].gt(jobs_clean["experience_max"]),
    ["source", "title", "company", "experience_raw", "experience_min", "experience_max", "url"],
]

display(jobs_clean["experience_bucket"].value_counts(dropna=False).rename_axis("experience_bucket").reset_index(name="jobs"))
display(jobs_clean["posted_week"].value_counts(dropna=False).sort_index().rename_axis("posted_week").reset_index(name="jobs"))
display(experience_anomalies)


,experience_bucket,jobs
0,1-3,531
1,3-5,321
2,unknown,197
3,0-1,190
4,0,104
5,5-10,84
6,10+,6


,posted_week,jobs
0,2026-W19,1
1,2026-W20,4
2,2026-W21,4
3,2026-W22,65
4,2026-W23,186
5,2026-W24,165
6,2026-W25,269
7,2026-W26,369
8,2026-W27,370


,source,title,company,experience_raw,experience_min,experience_max,url


## 5. Quality checks sau cleaning

Sau khi l?m s?ch, ta t?o b?ng ki?m tra ch?t l??ng v? assertion. N?u assertion fail, ngh?a l? rule cleaning v?a th?m ?ang l?m h?ng gi? ??nh quan tr?ng.

Pandas c?n ch? ?: `.agg()` ?? t?o summary theo source, lambda trong aggregation cho metric t?y ch?nh, v? `assert` ?? bi?n notebook th?nh t?i li?u c? ki?m ch?ng.


In [16]:
quality_summary = pd.DataFrame(
    [
        {"metric": "input_rows", "value": input_rows},
        {"metric": "rows_after_parse_ok_filter", "value": before_dedupe_rows},
        {"metric": "rows_after_url_dedupe", "value": len(jobs_clean)},
        {"metric": "unique_urls", "value": jobs_clean["url"].nunique()},
        {"metric": "duplicate_urls_after_cleaning", "value": int(jobs_clean["url"].duplicated().sum())},
        {"metric": "missing_title", "value": int(jobs_clean["title"].isna().sum())},
        {"metric": "missing_company", "value": int(jobs_clean["company"].isna().sum())},
        {"metric": "missing_description", "value": int(jobs_clean["description"].isna().sum())},
        {"metric": "numeric_salary_jobs", "value": int(jobs_clean["salary_label"].eq("numeric").sum())},
        {"metric": "numeric_suspicious_salary_jobs", "value": int(jobs_clean["salary_label"].eq("numeric_suspicious").sum())},
        {"metric": "jobs_with_skills", "value": int(jobs_clean["skill_count"].gt(0).sum())},
        {"metric": "jobs_with_cities", "value": int(jobs_clean["city_count"].gt(0).sum())},
    ]
)

quality_by_source = (
    jobs_clean.groupby("source", dropna=False)
    .agg(
        jobs=("url", "count"),
        unique_urls=("url", "nunique"),
        missing_title=("title", lambda s: int(s.isna().sum())),
        missing_company=("company", lambda s: int(s.isna().sum())),
        missing_description=("description", lambda s: int(s.isna().sum())),
        numeric_salary_jobs=("salary_label", lambda s: int(s.eq("numeric").sum())),
        suspicious_salary_jobs=("salary_label", lambda s: int(s.eq("numeric_suspicious").sum())),
        negotiable_salary_jobs=("salary_label", lambda s: int(s.eq("negotiable").sum())),
        hidden_salary_jobs=("salary_label", lambda s: int(s.eq("hidden").sum())),
        non_numeric_salary_label_jobs=("salary_label", lambda s: int(s.eq("non_numeric_label").sum())),
        jobs_with_skills=("skill_count", lambda s: int(s.gt(0).sum())),
        jobs_with_cities=("city_count", lambda s: int(s.gt(0).sum())),
    )
    .reset_index()
)
quality_by_source["numeric_salary_rate"] = quality_by_source["numeric_salary_jobs"] / quality_by_source["jobs"]
quality_by_source["skills_rate"] = quality_by_source["jobs_with_skills"] / quality_by_source["jobs"]
quality_by_source["cities_rate"] = quality_by_source["jobs_with_cities"] / quality_by_source["jobs"]

assert jobs_clean["url"].is_unique, "jobs_clean must contain one row per URL"
assert jobs_clean["parse_status"].eq("ok").all(), "jobs_clean should only contain parse_status == ok"
if input_rows == 1662 and jobs["url"].nunique() == 1433:
    assert len(jobs_clean) == 1433, "Current repository data should dedupe to 1,433 rows"
assert salary_clean["salary_suspicious_reason"].eq("").all(), "salary_clean must exclude suspicious salary rows"
assert job_skills["skill_norm"].fillna("").astype(str).str.strip().ne("").all(), "job_skills must not contain empty normalized skills"
assert job_cities["city"].fillna("").astype(str).str.strip().ne("").all(), "job_cities must not contain empty cities"

display(quality_summary)
display(quality_by_source.sort_values("source"))


,metric,value
0,input_rows,1662
1,rows_after_parse_ok_filter,1662
2,rows_after_url_dedupe,1433
3,unique_urls,1433
4,duplicate_urls_after_cleaning,0
5,missing_title,0
6,missing_company,0
7,missing_description,0
8,numeric_salary_jobs,504
9,numeric_suspicious_salary_jobs,8


,source,jobs,unique_urls,missing_title,missing_company,missing_description,numeric_salary_jobs,suspicious_salary_jobs,negotiable_salary_jobs,hidden_salary_jobs,non_numeric_salary_label_jobs,jobs_with_skills,jobs_with_cities,numeric_salary_rate,skills_rate,cities_rate
0,itviec,750,750,0,0,0,202,5,12,0,531,750,750,0.269333,1.000000,1.0
1,topcv,583,583,0,0,0,302,3,229,0,0,454,583,0.518010,0.778731,1.0
2,topdev,100,100,0,0,0,0,0,0,100,0,100,100,0.000000,1.000000,1.0


## 6. Export analysis datasets

C?c file export n?m trong `data/analysis` ?? kh?ng overwrite raw/processed data. ??y l? dataset ph?c v? h?c pandas v? ph?n t?ch ti?p theo; n?u rule n?o ?n ??nh, b??c sau m?i chuy?n v? parser.

Pandas c?n ch? ?: `.to_json(..., lines=True)` cho JSONL, `.to_csv(..., encoding="utf-8-sig")` ?? Excel ??c Unicode t?t h?n.


In [17]:
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

jobs_analysis_export = jobs_clean.drop(columns=["_salary_raw_folded", "_scraped_at_sort"], errors="ignore")

export_paths = {
    "jobs_analysis_clean_jsonl": ANALYSIS_DIR / "jobs_analysis_clean.jsonl",
    "jobs_analysis_clean_csv": ANALYSIS_DIR / "jobs_analysis_clean.csv",
    "job_skills_csv": ANALYSIS_DIR / "job_skills.csv",
    "job_cities_csv": ANALYSIS_DIR / "job_cities.csv",
    "salary_analysis_clean_csv": ANALYSIS_DIR / "salary_analysis_clean.csv",
    "quality_summary_csv": ANALYSIS_DIR / "quality_summary.csv",
    "quality_by_source_csv": ANALYSIS_DIR / "quality_by_source.csv",
}

jobs_analysis_export.to_json(
    export_paths["jobs_analysis_clean_jsonl"],
    orient="records",
    lines=True,
    force_ascii=False,
    date_format="iso",
)
jobs_analysis_export.to_csv(export_paths["jobs_analysis_clean_csv"], index=False, encoding="utf-8-sig")
job_skills.to_csv(export_paths["job_skills_csv"], index=False, encoding="utf-8-sig")
job_cities.to_csv(export_paths["job_cities_csv"], index=False, encoding="utf-8-sig")
salary_clean.to_csv(export_paths["salary_analysis_clean_csv"], index=False, encoding="utf-8-sig")
quality_summary.to_csv(export_paths["quality_summary_csv"], index=False, encoding="utf-8-sig")
quality_by_source.to_csv(export_paths["quality_by_source_csv"], index=False, encoding="utf-8-sig")

export_audit = pd.DataFrame(
    [
        {"dataset": name, "path": str(path.relative_to(REPO_ROOT)), "exists": path.exists(), "bytes": path.stat().st_size if path.exists() else 0}
        for name, path in export_paths.items()
    ]
)
display(export_audit)


,dataset,path,exists,bytes
0,jobs_analysis_clean_jsonl,data\analysis\jobs_analysis_clean.jsonl,True,7368216
1,jobs_analysis_clean_csv,data\analysis\jobs_analysis_clean.csv,True,6273641
2,job_skills_csv,data\analysis\job_skills.csv,True,1893727
3,job_cities_csv,data\analysis\job_cities.csv,True,1306908
4,salary_analysis_clean_csv,data\analysis\salary_analysis_clean.csv,True,1927215
5,quality_summary_csv,data\analysis\quality_summary.csv,True,310
6,quality_by_source_csv,data\analysis\quality_by_source.csv,True,482
